In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp, max as _max

# ------------------------------
# 1️ Define tables and consumer
# ------------------------------
SOURCE_TABLE = "fq_dev_catalog.bronze.sales_bills_2"
SILVER_TABLE = "fq_dev_catalog.silver.sales_bills"
VERSION_TABLE = "fq_dev_catalog.bronze.version_control"
CONSUMER_NAME = "sales_bills"                          # unique per notebook/consumer

# ------------------------------
# 2️ Get latest version from source Delta table
# ------------------------------
delta_table = DeltaTable.forName(spark, SOURCE_TABLE)

latest_version = (
    delta_table.history()
    .select(_max("version").alias("latest_version"))
    .collect()[0]["latest_version"]
)

# ------------------------------
# 3️ Get last processed version from VERSION_TABLE
# ------------------------------
if spark.catalog.tableExists(VERSION_TABLE):
    last_processed_version = (
        spark.table(VERSION_TABLE)
        .filter(col("source_table") == SOURCE_TABLE)
        .filter(col("consumer_name") == CONSUMER_NAME)
        .agg(_max("last_processed_version").alias("last_version"))
        .collect()[0]["last_version"]
    )
else:
    last_processed_version = None

last_processed_version = last_processed_version if last_processed_version is not None else -1

# ------------------------------
# 4️ Determine start_version safely
# ------------------------------
if last_processed_version >= latest_version:
    print(f"No new versions to process. Latest: {latest_version}, Last processed: {last_processed_version}")
    start_version = None
else:
    start_version = last_processed_version + 1
    print(f"Processing versions from {start_version} to {latest_version}")

In [0]:
from pyspark.sql.functions import col
 
SOURCE_TABLE = "fq_dev_catalog.bronze.sales_bills_2"
 
# start_version and latest_version are assumed to be computed in previous cell
# start_version = last_processed_version + 1

# Check if there are new versions to process
if start_version is None:
    # No new data to process - create empty DataFrame with same schema as source table
    incremental_df = spark.table(SOURCE_TABLE).limit(0)
    print("No new versions to process. Using empty DataFrame.")
else:
    # 1️ Read incremental data using CDF
    incremental_df = (
        spark.read
        .format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", start_version)
        .option("endingVersion", latest_version)
        .table(SOURCE_TABLE)
    )
    
    # 2️ Optional: keep only INSERT records
    incremental_df = incremental_df.filter(col("_change_type") == "insert")
 
display(incremental_df)

In [0]:
# %sql
# TRUNCATE TABLE fq_dev_catalog.bronze.version_control

In [0]:
from pyspark.sql.functions import (
    col, lit, to_date, collect_list,
    current_timestamp, sha2, concat_ws
)
from pyspark.sql.types import (
    StructType, StructField,
    StringType, ArrayType,
    DateType, DecimalType,
    IntegerType, BooleanType
)
from pyspark.sql.functions import (
    col, lit, collect_list, to_date,
    coalesce, current_timestamp
)
from pyspark.sql.functions import (
    col, trim, lower, regexp_replace, broadcast, expr
)

In [0]:
def normalize(col_name):
    return lower(trim(regexp_replace(col(col_name), r"\s+", " ")))

In [0]:
from pyspark.sql.functions import col

df_dim_store_norm = (
    spark.read.table("fq_dev_catalog.silver.dim_store")
    .select(
        "deployment_name",   # only for normalization
        "company_id",
        "country_code"
    )
    .withColumn(
        "deployment_name_norm",
        normalize("deployment_name")
    )
    .drop("deployment_name")   #  DROP IT AFTER NORMALIZATION
    .dropDuplicates(["deployment_name_norm"])
)


In [0]:
df_sales = (
    incremental_df
    .filter(col("ng_id").isNotNull())
    .dropDuplicates(["ng_id"])   # enable if needed
    .filter( (col("isVoid") == False) | col("isVoid").isNull() )
)

df_sales_clean = df_sales.withColumn(
    "deployment_name_norm",
    normalize("deployment_name")
)

df_sales_with_store = (
    df_sales_clean.alias("s")
    .join(
        df_dim_store_norm.alias("d"),
        col("s.deployment_name_norm") == col("d.deployment_name_norm"),  
        "left"
    )
)

In [0]:
# null_store_cnt = df_sales_with_store.filter(col("store_id").isNull()).count()
# print(f"Unmapped stores: {null_store_cnt}")

In [0]:
# df_sales_with_store.count()

In [0]:
from pyspark.sql.functions import expr
df_sales_with_kots = (
    df_sales_with_store
    .withColumn(
        "kots_ids",
        expr("""
            CASE
                WHEN _kots IS NOT NULL THEN
                    array_distinct(transform(_kots, k -> k.kotNumber))
                ELSE NULL
            END
        """)
    )
)

In [0]:
df_sales_with_kots.count()

In [0]:
df_bill = (
    df_sales_with_kots
    .withColumn("bill_id", col("billNumber_updated"))
    .withColumn("business_date", to_date(col("businessDate")))
    .withColumn("order_type", col("tabType"))
    .withColumn("tab_name", col("tab"))
    .withColumn("is_void", col("isVoid"))
    .withColumn("void_reason", col("voidComment"))
    .withColumn("source_system", lit("POSIST"))

    # NEW COLUMNS
    .withColumn("deployment_name", col("deployment_name"))
    .withColumn("close_time", col("_closeTime"))
    .withColumn("covers", col("_covers"))
    .withColumn("created", col("_created"))
)

In [0]:
dim_owned_cols = [
    "company_id",
    "country_code"
]

df_bill_clean = df_bill.drop(
    *[c for c in dim_owned_cols if c in df_bill.columns]
)

In [0]:
def normalize(c):
    return lower(trim(regexp_replace(c, r"\s+", " ")))
    
df_bill_enriched = (
    df_bill_clean.alias("f")
    .join(
        df_dim_store_norm.alias("d"),
        (normalize(col("f.deployment_name")) == col("d.deployment_name_norm")),
        "left"
    )
)

In [0]:
# df_bill_enriched.count()

In [0]:
df_sales_bill_final = df_bill_enriched.select(

    # ---- SURROGATE KEY ----
    expr("uuid()").alias("sys_id"),

    # ---- BUSINESS KEYS ----
    col("bill_id"),
    col("ng_id"),
    col("kots_ids"),
    col("business_date"),

    # ---- DIM STORE ----
    col("deployment_name"),
    col("company_id"),
    col("country_code"),

    # ---- AMOUNTS ----
    col("aggregation.totalAmount")
        .cast(DecimalType(18,3))
        .alias("net_amount_before_discount"),

    col("aggregation.billDiscount")
        .cast(DecimalType(18,3))
        .alias("bill_discount"),

    (
        col("aggregation.totalAmount")
        - coalesce(col("aggregation.billDiscount"), lit(0))
        - coalesce(col("aggregation.totalItemDiscount"), lit(0))
    )
    .cast(DecimalType(18,3))
    .alias("net_amount_after_discount"),

    col("aggregation.totalTax")
        .cast(DecimalType(18,3))
        .alias("tax_amount"),

    col("aggregation.netAmount")
        .cast(DecimalType(18,3))
        .alias("gross_amount"),

    col("aggregation.totalItems")
        .cast(IntegerType())
        .alias("total_items"),

    col("aggregation.totalItemDiscount")
        .cast(DecimalType(18,3))
        .alias("total_items_discount"),

    # ---- NEW FIELDS ----
    col("close_time"),
    col("covers").cast(IntegerType()).alias("covers"),
    col("created"),

    # ---- FLAGS ----
    col("is_void"),
    col("void_reason"),
    col("order_type"),
    col("tab_name"),
    col("source_system")
)

In [0]:
df_sales_bill_final = df_sales_bill_final.filter(
    (col("is_void") == False) | col("is_void").isNull()
)

In [0]:
df_sales_bill_final.display()

In [0]:
df_sales.count(), df_sales_bill_final.count()

In [0]:
# df_sales_bill_final.limit(0).write \
#     .format("delta") \
#     .mode("overwrite") \
#     .saveAsTable("fq_dev_catalog.silver.sales_bills")

In [0]:
# %sql
# ALTER TABLE fq_dev_catalog.silver.sales_bills
# SET TBLPROPERTIES (
#   'delta.enableChangeDataFeed' = 'true',
#   'delta.columnMapping.mode' = 'name'
# )

In [0]:
# %sql
# MERGE INTO fq_dev_catalog.silver.sales_bills AS tgt
# USING (
#   SELECT
#     ng_id,
#     _created AS created
#   FROM fq_dev_catalog.bronze.sales_bills_2
#   WHERE ng_id IS NOT NULL
#   GROUP BY ng_id, _created
# ) src
# ON tgt.ng_id = src.ng_id
# WHEN MATCHED THEN
#   UPDATE SET
#     tgt.created = src.created

In [0]:
# from delta.tables import DeltaTable

# TARGET_TABLE = "fq_dev_catalog.silver.sales_bills"

# target = DeltaTable.forName(spark, TARGET_TABLE)

# (
#     target.alias("tgt")
#     .merge(
#         df_sales_bill_final.alias("src"),
#         """
#         tgt.ng_id = src.ng_id
        
#         """
#     )
#     .whenMatchedUpdateAll()
#     .whenNotMatchedInsertAll()
#     .execute()
# )

# print("Merge completed successfully")


In [0]:
from delta.tables import DeltaTable

TARGET_TABLE = "fq_dev_catalog.silver.sales_bills"

target = DeltaTable.forName(spark, TARGET_TABLE)

# Get distinct business keys from incremental batch
keys_df = df_sales_bill_final.select("ng_id").distinct()

# Delete matching rows in silver
(
    target.alias("tgt")
    .merge(
        keys_df.alias("src"),
        "tgt.ng_id = src.ng_id"
    )
    .whenMatchedDelete()
    .execute()
)

print("Deleted existing silver rows for incoming ng_id(s)")


In [0]:
df_sales_bill_final.write.format("delta") \
    .mode("append") \
    .saveAsTable(TARGET_TABLE)

print("Inserted fresh rows into silver")

In [0]:

from pyspark.sql.functions import current_timestamp

SOURCE_TABLE = "fq_dev_catalog.bronze.sales_bills_2"
CONSUMER_NAME = "sales_bills"
VERSION_TABLE = "fq_dev_catalog.bronze.version_control"

spark.sql(f"""
UPDATE {VERSION_TABLE}
SET
  last_processed_version = {latest_version},
  updated_at = current_timestamp()
WHERE
  source_table = '{SOURCE_TABLE}'
  AND consumer_name = '{CONSUMER_NAME}'
""")

In [0]:
# %sql
# SELECT * FROM fq_dev_catalog.bronze.version_control;

In [0]:
# %sql
# MERGE INTO fq_dev_catalog.silver.sales_bills AS tgt
# USING (
#   SELECT
#     deployment_name,
#     company_id
#   FROM fq_dev_catalog.silver.dim_store
# ) src
# ON tgt.deployment_name = src.deployment_name
# WHEN MATCHED THEN
#   UPDATE SET
#     tgt.company_id = src.company_id;